# Code for horizontal_distribution.ipynb

### Install and Import the necessary classes from the RDFlib library:

In [1]:
! pip install pandas
! pip install openpyxl
!pip install rdflib

import pandas as pd
import rdflib
import hashlib
import time
import os
import random
import numpy as np
import urllib.parse
from rdflib import Literal, Namespace, RDF, URIRef
from rdflib.namespace import FOAF, XSD
from rdflib import Graph, Namespace, RDF, RDFS, OWL
from rdflib.plugins.sparql import prepareQuery
from pyspark.sql.functions import when, col, lit






Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


### Explanation:
Cattle: 5 RDF files (~27.35 MB)

Pig: 6 RDF files (~9.87 MB)

Poultry: 1 RDF file (~1.53 MB)

Total Data Volume: ~38.75 MB

##### Objective
Distribute this total dataset evenly across 6 labs, so that each lab holds a fair mix of all three domains (cattle, pig, poultry), preserving overall proportions.

Let’s fix lab size to ≈ 6.46 MB per lab (38.75 ÷ 6)

##### Process
All TTL (RDF) files were loaded and merged based on domain.

The combined RDF graph (including all triples from cattle, pig, and poultry) was randomly shuffled to remove any ordering bias.

The merged dataset was then divided into 6 equal parts, each written to a separate TTL file:

HorizontalLab1.ttl to HorizontalLab6.ttl

Each lab thus contains approximately 1/6 of the total triples, representing a balanced sample of the entire dataset.
##### Outcome
Each lab pod now holds a proportionate blend of cattle, pig, and poultry data.

This setup enables testing how the federated query engine scales and performs when data is horizontally sharded across multiple sources — a common real-world scenario in distributed data environments.MB  

In [2]:
# Step 1: File paths
# Define file paths for cattle, pig, and poultry RDF datasets to be loaded for distribution. For this experiment, we put TTL files in the data folder.

cattle_files = [
    "https://solidserver.bovi-analytics.com/decide_lab1/Vertical/RDFoutputCattleSampleLab1.ttl",
    "https://solidserver.bovi-analytics.com/decide_lab2/Vertical/RDFoutputCattleSampleLab2.ttl",
    "https://solidserver.bovi-analytics.com/decide_lab3/Vertical/RDFoutputCattleSampleLab3.ttl",
    "https://solidserver.bovi-analytics.com/decide_lab4/Vertical/RDFoutputCattleSampleLab4.ttl",
    "https://solidserver.bovi-analytics.com/decide_lab5/Vertical/RDFoutputCattleSampleLab5.ttl"
]

pig_files = [
    "https://solidserver.bovi-analytics.com/decide_lab6/Vertical/PigLab1.ttl",
    "https://solidserver.bovi-analytics.com/decide_lab7/Vertical/PigLab2.ttl",
    "https://solidserver.bovi-analytics.com/decide_lab8/Vertical/PigLab3.ttl",
    "https://solidserver.bovi-analytics.com/decide_lab9/Vertical/PigLab4.ttl",
    "https://solidserver.bovi-analytics.com/decide_lab10/Vertical/PigLab5.ttl",
    "https://solidserver.bovi-analytics.com/decide_lab11/Vertical/PigLab6.ttl"
]

poultry_file = "https://solidserver.bovi-analytics.com/decide_lab12/Vertical/Poultrylab1.ttl"


In [3]:
# Step 2: Load and merge all RDFs
def load_graphs(file_list):
    g = Graph()
    for file in file_list:
        g.parse(file, format="ttl")
    return g

g_cattle = load_graphs(cattle_files)
g_pig = load_graphs(pig_files)
g_poultry = Graph().parse(poultry_file, format="ttl")

g_combined = g_cattle + g_pig + g_poultry
print(f" Total merged triples: {len(g_combined)}")

 Total merged triples: 3262


In [4]:
# Step 3: Group triples by subject
from collections import defaultdict
subject_blocks = defaultdict(list)
for s, p, o in g_combined:
    subject_blocks[s].append((s, p, o))

print(f" Total unique subjects (samples/entities): {len(subject_blocks)}")

 Total unique subjects (samples/entities): 1178


In [5]:
# Step 4:  Shuffle and split into 6 labs
num_labs = 6
all_subjects = list(subject_blocks.keys())
random.shuffle(all_subjects)

lab_subjects = [all_subjects[i::num_labs] for i in range(num_labs)]

# Write TTL files for each lab
output_dir = "horizontal_output"
os.makedirs(output_dir, exist_ok=True)

for i, subject_list in enumerate(lab_subjects):
    g_lab = Graph()
    for subj in subject_list:
        for triple in subject_blocks[subj]:
            g_lab.add(triple)
    
    output_path = os.path.join(output_dir, f"HorizontalLab{i+1}.ttl")
    g_lab.serialize(destination=output_path, format="ttl")
    print(f" Written {output_path} with {len(g_lab)} triples")

 Written horizontal_output\HorizontalLab1.ttl with 603 triples
 Written horizontal_output\HorizontalLab2.ttl with 510 triples
 Written horizontal_output\HorizontalLab3.ttl with 519 triples
 Written horizontal_output\HorizontalLab4.ttl with 525 triples
 Written horizontal_output\HorizontalLab5.ttl with 549 triples
 Written horizontal_output\HorizontalLab6.ttl with 556 triples


In [6]:
#To check if during data distribution, data is correctly distributed and semantic maintained, after that these ttls are ready to upload on solid pod


from rdflib import Graph

# Load local RDF TTL file
g = Graph()
g.parse("horizontal_output/HorizontalLab1.ttl", format="ttl")

# SPARQL query — simplified cattle example
q = """
PREFIX LHO: <http://www.purl.org/decide/LiveStockHealthOnto/LHO#>

SELECT ?Sample ?Breed ?Pathogen ?SampleType ?SampleResult
WHERE {
    ?Sample a LHO:CattleSample ;
            LHO:hasBreed ?Breed ;
            LHO:hasPathogen ?Pathogen ;
            LHO:hasSampleType ?SampleType ;
            LHO:hasResult ?SampleResult .
}
LIMIT 10
"""

# Run the query
results = g.query(q)

# Display results
for row in results:
    print([str(cell).split("#")[-1] for cell in row])


['Lab1CattleSample_100', 'Beef', 'MB', 'Swab', '1']
['Lab2CattleSample_51164', 'Dairy', 'PI3', 'BAL', '0']
['Lab2CattleSample_51168', 'Dairy', 'PM', 'BAL', 'Missing']
['Lab3CattleSample_24448', 'Beef', 'PM', 'BAL', '0']
['Lab3CattleSample_24454', 'Beef', 'BCV', 'BAL', 'Missing']
['Lab3CattleSample_24457', 'Beef', 'HS', 'BAL', '1']
['Lab4CattleSample_75700', 'Unknown', 'PM', 'BAL', '0.0']
['Lab4CattleSample_75703', 'Unknown', 'MB', 'BAL', '0.0']
['Lab4CattleSample_75709', 'Unknown', 'HS', 'BAL', '0.0']
['Lab5CattleSample_66604', 'Beef', 'PM', 'Autopsy', '1']


In [7]:
g

<Graph identifier=Nf5f3ef949c7f45b3a75068aeed204a27 (<class 'rdflib.graph.Graph'>)>